# Limpieza de hospitales.csv

Catálogo de hospitales: id, nombre, nombre corto, distrito o municipio, tipo (público, privado o concertado), camas, urgencias 24h y helipuerto.

El CSV trae 20 filas, una por hospital. Al final guardamos `hospitales_limpio.csv`, también con 20 filas: aquí no hay filas que quitar, solo valores que poner en el mismo formato.

El CSV original no se toca: el resultado se guarda aparte en `hospitales_limpio.csv`.

In [1]:
import pandas as pd
import numpy as np
import unicodedata
from pathlib import Path

# El CSV original debe estar junto al notebook o en la carpeta data del repo.
ruta = Path('hospitales.csv')
if not ruta.exists():
    ruta = Path('../data/hospitales.csv')
hosp = pd.read_csv(ruta)
hosp.head()

,hospital_id,nombre,nombre_corto,distrito,tipo,camas,urgencias_24h,helipad
0,1,Hospital Universitario La Paz,LA PAZ,FUENCARRAL-EL PARDO,Público,1328.0,1,1
1,2,Hospital General Universitario Gregorio Marañón,GREGORIO MARAÑON,RETIRO,Público,1500.0,1,0
2,3,Hospital Clínico San Carlos,clinico san carlos,MONCLOA-ARAVACA,PRIVADO,961.0,1,0
3,4,Hospital Universitario 12 de Octubre,12 de Octubre,USERA,Público,1256.0,1,1
4,5,Hospital Universitario Ramón y Cajal,Ramón y Cajal,HORTALEZA,publico,1092.0,1,1


## Revisión inicial

Miramos los datos antes de tocar nada: cuántas filas hay, dónde están los huecos y qué valores raros aparecen.

In [2]:
print(hosp.shape)             # 20 filas, una por hospital
print()
print(hosp.isna().sum())      # huecos en camas
print()
print(f"hospital_id repetidos: {hosp['hospital_id'].duplicated().sum()}")
print(f"Filas duplicadas enteras: {hosp.duplicated().sum()}")
print()
print("Tipos escritos de distintas formas:")
print(sorted(hosp['tipo'].unique()))
print()
print(f"Camas min/max: {hosp['camas'].min()} / {hosp['camas'].max()}")
print(f"urgencias_24h: {sorted(hosp['urgencias_24h'].unique())} | helipad: {sorted(hosp['helipad'].unique())}")

(20, 8)

hospital_id      0
nombre           0
nombre_corto     0
distrito         0
tipo             0
camas            3
urgencias_24h    0
helipad          0
dtype: int64

hospital_id repetidos: 0
Filas duplicadas enteras: 0

Tipos escritos de distintas formas:
['PRIVADO', 'PUBLICO', 'Privado', 'Público', 'privado-concertado', 'publico']

Camas min/max: -50.0 / 1500.0
urgencias_24h: [np.int64(0), np.int64(1)] | helipad: [np.int64(0), np.int64(1)]


## Tipo de hospital y nombre corto

El tipo viene escrito con mayúsculas, minúsculas y tildes mezcladas (`Público`, `PRIVADO`, `publico`). Solo cambia la escritura, no el significado: con la misma regla de texto de siempre quedan las 3 categorías reales (`PUBLICO`, `PRIVADO`, `PRIVADO-CONCERTADO`). No hace falta ninguna equivalencia inventada.

El nombre corto pasa al mismo formato (mayúsculas, sin tildes) para que no aparezca el mismo hospital escrito de dos formas (`clinico san carlos`).

In [3]:
def normalizar_texto(s):
    if pd.isna(s):
        return None
    s = unicodedata.normalize('NFD', str(s).strip().upper())
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    return ' '.join(s.split())

hosp['tipo'] = hosp['tipo'].map(normalizar_texto)
hosp['nombre_corto'] = hosp['nombre_corto'].map(normalizar_texto)
print(sorted(hosp['tipo'].unique()))
print()
print(hosp['tipo'].value_counts())

['PRIVADO', 'PRIVADO-CONCERTADO', 'PUBLICO']

tipo
PUBLICO               15
PRIVADO-CONCERTADO     3
PRIVADO                2
Name: count, dtype: int64


## Distrito o municipio

Aplicamos la misma regla de nombres que en las otras limpiezas: mayúsculas, sin tildes, sin el artículo del principio y, si el texto es el principio de un único distrito oficial, se completa.

Aquí la regla deja dos grupos que no son un distrito de Madrid y no se fuerzan:

- **Hospitales de otros municipios** (Getafe, Móstoles, Alcorcón...): el dato es real, el hospital está ahí, así que se conserva el nombre del municipio. Es la misma situación que en `accidentes.csv`: fuera del término municipal no hay datos socioeconómicos con los que cruzar, y en `cruces_datasets.ipynb` estos hospitales ya quedan fuera de la oferta por distrito.
- **Hospital Infanta Leonor (`VALLECAS` a secas):** no sabemos si es Puente o Villa de Vallecas, y un hospital no sale de una base del SAMUR, así que no se puede aplicar el truco de las unidades. Se deja como `VALLECAS`, sin asignar, como ya pasó con las activaciones.

In [4]:
OFICIALES = ["CENTRO", "ARGANZUELA", "RETIRO", "SALAMANCA", "CHAMARTIN", "TETUAN", "CHAMBERI",
             "FUENCARRAL-EL PARDO", "MONCLOA-ARAVACA", "LATINA", "CARABANCHEL", "USERA",
             "PUENTE DE VALLECAS", "MORATALAZ", "CIUDAD LINEAL", "HORTALEZA", "VILLAVERDE",
             "VILLA DE VALLECAS", "VICALVARO", "SAN BLAS-CANILLEJAS", "BARAJAS"]

def normalizar_distrito(nombre):
    t = normalizar_texto(nombre)
    if t is None:
        return None
    for articulo in ('LA ', 'EL ', 'LOS ', 'LAS '):
        if t.startswith(articulo):
            t = t[len(articulo):]
    if t in OFICIALES:
        return t
    candidatos = [o for o in OFICIALES if o.startswith(t)]
    if len(candidatos) == 1:
        return candidatos[0]
    return t  # no es un distrito de Madrid: se conserva el texto normalizado (municipio o VALLECAS)

hosp['distrito'] = hosp['distrito'].map(normalizar_distrito)
fuera = hosp[~hosp['distrito'].isin(OFICIALES)]
print(f"Hospitales sin distrito de Madrid: {len(fuera)} de {len(hosp)}")
print(fuera[['hospital_id', 'nombre_corto', 'distrito']].to_string(index=False))

Hospitales sin distrito de Madrid: 10 de 20
 hospital_id         nombre_corto                   distrito
           9            H. GETAFE                     GETAFE
          10         SEVERO OCHOA                    LEGANES
          11 PRINCIPE DE ASTURIAS          ALCALA DE HENARES
          12          H. MOSTOLES                   MOSTOLES
          13       INFANTA LEONOR                   VALLECAS
          14       H. DEL HENARES                    COSLADA
          15        INFANTA SOFIA SAN SEBASTIAN DE LOS REYES
          16      REY JUAN CARLOS                   MOSTOLES
          17     INFANTA CRISTINA                      PARLA
          20          H. ALCORCON                   ALCORCON


## Camas

Cinco hospitales se quedan sin número de camas:

- Tres ya venían vacíos (Fundación Jiménez Díaz, Príncipe de Asturias y H. Móstoles).
- Dos traían -50, un valor imposible (no puede haber camas negativas): pasa a vacío.

No inventamos el dato: el hueco se conserva. Cuando hace falta un número para los cruces, `cruces_datasets.ipynb` ya lo trata allí (a un hospital que sale en el catálogo sin camas no se le cuentan 0 camas).

In [5]:
print("Camas imposibles (<= 0):")
print(hosp.loc[hosp['camas'] <= 0, ['hospital_id', 'nombre_corto', 'camas']].to_string(index=False))

hosp.loc[hosp['camas'] <= 0, 'camas'] = np.nan
hosp['camas'] = hosp['camas'].astype('Int64')  # entero que admite vacíos
print()
print(f"Hospitales sin dato de camas: {hosp['camas'].isna().sum()}")
print(f"Camas min/max tras limpiar: {hosp['camas'].min()} / {hosp['camas'].max()}")

Camas imposibles (<= 0):
 hospital_id    nombre_corto  camas
          16 REY JUAN CARLOS  -50.0
          20     H. ALCORCON  -50.0

Hospitales sin dato de camas: 5
Camas min/max tras limpiar: 150 / 1500


## Estadísticos y guardado

Las 20 filas quedan ordenadas por `hospital_id`. `urgencias_24h` y `helipad` ya venían bien (solo 0 y 1) y no se tocan. Miramos los estadísticos finales y guardamos.

Ojo al cruzar con `distritos.csv`: su columna `num_hospitales` no coincide con lo que sale de contar este catálogo en varios distritos. No corregimos ninguna de las dos fuentes: la comprobación está en `cruces_datasets.ipynb`.

In [6]:
hosp = hosp.sort_values('hospital_id').reset_index(drop=True)
# Mismo orden de columnas que el original
hosp = hosp[['hospital_id', 'nombre', 'nombre_corto', 'distrito', 'tipo',
             'camas', 'urgencias_24h', 'helipad']]
hosp.to_csv('../data_clean/hospitales_limpio.csv', index=False)
print(hosp.shape)   # (20, 8)
print()
print("Huecos que quedan (los valores que no conocemos):")
print(hosp.isna().sum()[hosp.isna().sum() > 0])
print()
hosp.describe().round(2)

(20, 8)

Huecos que quedan (los valores que no conocemos):
camas    5
dtype: int64



,hospital_id,camas,urgencias_24h,helipad
count,20.00,15.0,20.00,20.00
mean,10.50,662.07,0.95,0.35
std,5.92,446.75,0.22,0.49
min,1.00,150.0,0.00,0.00
25%,5.75,296.5,1.00,0.00
50%,10.50,500.0,1.00,0.00
75%,15.25,1026.5,1.00,1.00
max,20.00,1500.0,1.00,1.00
